## 3.3 Logistic Regression

**Wat is logistic regression?**
Logistic regression is een classificatiemodel. Het model berekent eerst een gewogen som van de features:

$$z = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_n x_n$$

Deze som zet het model met de sigmoïdefunctie om naar een kans tussen 0 en 1:

$$P(\text{stroke} = 1) = \frac{1}{1 + e^{-z}}$$

Is de kans groter dan de threshold (standaard 0,5), dan voorspelt het model een herseninfarct.

**Waarom dit model?**
- Het is een eenvoudig en snel model, en daarom een goede baseline om andere modellen mee te vergelijken.
- De coëfficiënten zijn goed te interpreteren: een positieve coëfficiënt betekent dat de feature de kans op een herseninfarct verhoogt. Dat is belangrijk voor artsen, die willen begrijpen waarom het model een patiënt als risicopatiënt aanwijst.

**Data inladen**
We laden de opgeschoonde data in via `StrokeData`. Deze voert dezelfde opschoonstappen uit als in paragraaf 1.6 van de EDA.

In [1]:
"""Logistic regression voor de Kaggle-competitie Predicting stroke."""
import sys

sys.path.append("..")  # zodat de map src gevonden wordt

from src.preprocessing import NUMERIC_COLS, StrokeData

data = StrokeData()
X, y = data.get_xy()
X_test = data.get_test()

print(X.shape, y.shape, X_test.shape)
print("Percentage stroke:", round(y.mean() * 100, 2))

(33544, 15) (33544,) (8388, 15)
Percentage stroke: 1.54


De trainset bevat 33.544 patiënten en 15 features. Slechts 1,54% heeft een herseninfarct gehad.

--------
### Baseline

We beginnen met een logistic regression met standaardinstellingen. Dit dient als baseline: elk volgend model moet beter scoren.

- Alleen de numerieke features worden gestandaardiseerd (zie paragraaf 1.7). De binaire features blijven 0/1.
- De scaler zit in een `Pipeline`, zodat hij per fold alleen op de trainingsdata wordt gefit. Zo voorkomen we data leakage.
- We evalueren met 5-fold `StratifiedKFold`: elke fold bevat hetzelfde percentage herseninfarcten (1,54%).

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    [("scale", StandardScaler(), NUMERIC_COLS)],
    remainder="passthrough",  # binaire kolommen blijven 0/1
)

baseline = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
}

scores = cross_validate(baseline, X, y, cv=cv, scoring=scoring)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")

accuracy: 0.985
precision: 0.000
recall: 0.000
f1: 0.000


**Observatie:** de baseline voorspelt voor geen enkele patiënt een herseninfarct. De accuracy is 0,985, maar dat is precies het percentage patiënten zonder herseninfarct. De recall en F1-score zijn 0: het model vindt geen enkele risicopatiënt en is daarmee voor een arts onbruikbaar. Door de sterke onbalans is het voor het model 'voordeliger' om de kleine klasse te negeren.